In [1]:
# !pip install -r requirements.txt -qqq

In [2]:
import numpy as np
import pandas as pd
import warnings
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import torch

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

%env JOBLIB_TEMP_FOLDER=/tmp

True
NVIDIA GeForce RTX 3060
env: JOBLIB_TEMP_FOLDER=/tmp


In [3]:
folder_path = 'dataset/'
train_identity = pd.read_csv(f'{folder_path}train_identity.csv')
train_transaction = pd.read_csv(f'{folder_path}train_transaction.csv')
test_identity = pd.read_csv(f'{folder_path}test_identity.csv')
test_transaction = pd.read_csv(f'{folder_path}test_transaction.csv')

# let's combine the data and work with the whole dataset
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

# train.to_csv(f'{folder_path}merged_train.csv')
RANDOM_SEED = 42
START_DATE = "2026-01-01"


In [4]:
del train_identity, train_transaction, test_identity, test_transaction

In [5]:
print(train.columns.tolist())

['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V

In [6]:
train['isFraud'].value_counts(normalize=True)

isFraud
0    0.965009990855827
1    0.034990009144173
Name: proportion, dtype: float64

In [7]:
# columns grouping
# transaction columns
transaction_cols = ["TransactionID", "TransactionDT", "TransactionAmt"]
# product columns
product_cols = ["ProductCD"]
# card columns
card_cols = ["card1", "card2", "card3", "card4", "card5", "card6"]
# address columns
address_cols = ["addr1", "addr2"]
# email columns
email_cols = ["P_emaildomain", "R_emaildomain"]
# distance columns
distance_cols = ["dist1", "dist2"]
# matching features columns
matching_cols = ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"]
# count columns
count_cols = [f"C{i}" for i in range(1, 15)]
# delay columns
delay_cols = [f"D{i}" for i in range(1, 16)]
# v columns
v_cols = [f"V{i}" for i in range(1, 340)]
# device columns
device_cols = ["DeviceType", "DeviceInfo"]
# identity columns
identity_cols = [f"id_{str(i).zfill(2)}" for i in range(1, 39)]
# device metadata columns 
# id_30: Operating System, 
# id_31: Browser
# id_32: screen color depth
# id_33: screen resolution
device_metadata_cols = ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_32", "id_33"]

In [8]:
check = train[["TransactionID", "TransactionDT", "TransactionAmt", "DeviceType", "DeviceInfo", "id_30", "id_31", "id_32", "id_33", "isFraud"]]
check[check["isFraud"] == True]

,TransactionID,TransactionDT,TransactionAmt,DeviceType,DeviceInfo,id_30,id_31,id_32,id_33,isFraud
203,2987203,89760,445.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1
240,2987240,90193,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
243,2987243,90246,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
245,2987245,90295,37.097999999999999,mobile,Redmi Note 4 Build/MMB29M,NaN,chrome 54.0 for android,NaN,NaN,1
288,2987288,90986,155.520999999999987,mobile,NaN,NaN,chrome 62.0 for ios,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...
590361,3577361,15807368,1224.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1
590364,3577364,15807516,69.963999999999999,mobile,SAMSUNG SM-J700M Build/MMB29K,NaN,samsung browser 6.4,NaN,NaN,1
590368,3577368,15807677,100.000000000000000,mobile,iOS Device,iOS 11.3.0,mobile safari 11.0,32.0,2208x1242,1
590372,3577372,15807758,117.000000000000000,NaN,NaN,NaN,NaN,NaN,NaN,1


In [9]:
# Date feature engineering with TransactionDT
train["DT"] = pd.to_datetime(START_DATE) + pd.to_timedelta(
    train["TransactionDT"], unit="s"
)

# Extract temporal features
train["DT_month"] = train["DT"].dt.month
train["DT_week"] = train["DT"].dt.isocalendar().week.astype("int32")
train["DT_day"] = train["DT"].dt.day
train["DT_weekday"] = train["DT"].dt.weekday
train["DT_hour"] = train["DT"].dt.hour

# Optional: remove the temporary datetime column
train.drop(columns=["DT"], inplace=True)

In [10]:
# Construct UID
train["uid"] = (
    train["card1"].astype(str) + "_" +
    train["card2"].astype(str) + "_" +
    train["card3"].astype(str) + "_" +
    train["card5"].astype(str)
)

# Construct UID2
train["uid2"] = (
    train["uid"] + "_" +
    train["addr1"].astype(str) + "_" +
    train["P_emaildomain"].astype(str)
)

In [11]:
# reduce memory
def reduce_mem_usage(df, verbose=True):
    """
    Iterate through all columns of a dataframe and
    modify the data type to reduce memory usage.
    """

    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype

        # Skip datetime columns
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue

        # Integer columns
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)

        # Float columns
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)

        # Object columns
        elif col_type == object:
            df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2

    if verbose:
        print(f"Memory usage before: {start_mem:.2f} MB")
        print(f"Memory usage after : {end_mem:.2f} MB")
        print(f"Decreased by {(100 * (start_mem - end_mem) / start_mem):.1f}%")

    return df

train = reduce_mem_usage(train)

Memory usage before: 2666.48 MB
Memory usage after : 1747.37 MB
Decreased by 34.5%


In [12]:
def encode_categorical_columns(df):
    """
    Encode categorical columns for IEEE-CIS Fraud Detection.

    - M1-M3, M5-M9:
        T -> 1
        F -> 0
        NaN -> -1

    - M4:
        Label encoded (M0, M1, M2, Missing)

    - Remaining object/category columns:
        Label encoded.
    """

    # -------------------------
    # Handle M1-M3 and M5-M9
    # -------------------------
    matching_binary_cols = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]

    for col in matching_binary_cols:
        if col in df.columns:
            df[col] = df[col].replace({"T": 1, "F": 0}).fillna(-1).astype("int8")

    # -------------------------
    # Handle M4 separately
    # -------------------------
    if "M4" in df.columns:
        le = LabelEncoder()

        df["M4"] = df["M4"].fillna("Missing").astype(str)

        df["M4"] = le.fit_transform(df["M4"])

    # -------------------------
    # Label encode remaining
    # categorical columns
    # -------------------------
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:

        # M4 has already been encoded
        if col == "M4":
            continue

        le = LabelEncoder()

        df[col] = df[col].fillna("Missing").astype(str)

        df[col] = le.fit_transform(df[col])

    return df


train = encode_categorical_columns(train)

In [13]:
# Handle missing values
def handle_missing_values(df):
    # Numerical columns
    num_cols = df.select_dtypes(include=["int", "float"]).columns

    # Fill missing numerical values with -999
    df[num_cols] = df[num_cols].fillna(-999)

    # Object / categorical columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns

    # Preserve missing information as string
    df[cat_cols] = df[cat_cols].fillna("Missing")

    return df

train = handle_missing_values(train)

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

train = train.sort_values("TransactionDT")

# Drop target and TransactionID
X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]

# Time-based 80-20 split
split_idx = int(len(train) * 0.8)

X_train = X.iloc[:split_idx]
X_valid = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

In [15]:
# Random Forest
# Train model
rf_model = RandomForestClassifier()

rf_model.fit(X_train, y_train)

# Evaluate
y_pred_prob = rf_model.predict_proba(X_valid)[:, 1]

print("Random Forest")
print("ROC-AUC:", roc_auc_score(y_valid, y_pred_prob))

Random Forest
ROC-AUC: 0.8898522600632524


In [16]:
# LighGBM
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
)

lgb_model.fit(X_train, y_train)

# Fraud probabilities
y_pred_prob = lgb_model.predict_proba(X_valid)[:, 1]

# ROC-AUC score
roc_auc = roc_auc_score(y_valid, y_pred_prob)

print("LightGBM")
print(f"ROC-AUC: {roc_auc:.4f}")

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.646235 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35129
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 438
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
LightGBM
ROC-AUC: 0.9082
